**Machine Learning Lab**

Clustering and hierarchical clustering are both unsupervised learning techniques used to group similar data points together based on certain characteristics or patterns. However, they differ in approach and use cases.

# 11. Clustering

Clustering is the process of dividing a dataset into distinct groups, or clusters, where data points in the same cluster are more similar to each other than to those in other clusters.

**Popular Methods:**

1. **K-means**: Partitions data into K clusters by minimizing the variance within each cluster.
2. **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**: Groups data points based on density, which helps detect clusters of varying shapes and sizes. Also identifies outliers as noise.
3. **Gaussian Mixture Models (GMM)**: Assumes data is generated from a mixture of several Gaussian distributions and clusters by maximizing the likelihood of the data given these distributions.

**When to Use:** Clustering is useful for finding patterns in data when there are no predefined labels. It's used in customer segmentation, image compression, and document clustering, among other applications.

**Limitations:** Many clustering methods, like K-means, struggle with clusters that are non-spherical or vary in density and size. Additionally, choosing the right number of clusters can be challenging.

## Importing Libraries

In [ ]:
import pandas as pd

## Reading Dataset

This notebook uses `countries_of_the_world.csv`. Run the cell below and upload it (found in this notebook's folder in the repo) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select countries_of_the_world.csv

In [ ]:
df = pd.read_csv('countries_of_the_world.csv')

Preview the raw data.

In [ ]:
df.head()

## Preprocessing

Check for missing values in every column.

In [ ]:
df.isnull().sum()

**Normalization**

List every column, to identify which ones hold numeric data.

In [ ]:
df.columns

This dataset was originally formatted with **commas as decimal separators** (a common convention in many European countries) — e.g. `"0,38"` instead of `"0.38"` — so pandas read these columns in as text rather than numbers. We fix that here for every numeric column, then convert the result with `pd.to_numeric`. The `try/except` skips a column silently if the conversion doesn't apply to it (e.g. if it was already numeric).

In [ ]:
numeric_columns = ['Population', 'Area (sq. mi.)',
       'Pop. Density (per sq. mi.)', 'Coastline (coast/area ratio)',
       'Net migration', 'Infant mortality (per 1000 births)',
       'GDP ($ per capita)', 'Literacy (%)', 'Phones (per 1000)', 'Arable (%)',
       'Crops (%)', 'Other (%)', 'Birthrate', 'Deathrate',
       'Agriculture', 'Industry', 'Service']

# Iterate over each column in the DataFrame
for col in numeric_columns:
  try:
    df[col] = pd.to_numeric(df[col].str.replace(',', '.'))
  except:
    pass

df.head()

A few of these columns still have missing values (check `df[numeric_columns].isnull().sum()` — `'Literacy (%)'` alone is missing 18 out of 227 countries). The clustering algorithms we'll use below can't handle `NaN` values directly, so we fill them in with each column's mean before moving on. (Without this step, `DBSCAN` further down would fail with `ValueError: Input X contains NaN`.)

In [ ]:
# Impute missing numeric values with the column mean
df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].mean())

df[numeric_columns].isnull().sum()

Now scale every numeric column into the [0, 1] range with `MinMaxScaler`, so that no single feature (e.g. `Population`, which can be in the hundreds of millions) dominates the distance calculations that clustering algorithms rely on.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Create a MinMaxScaler object
scaler = MinMaxScaler()

# Fit and transform the selected columns
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])

df.head()

## Clustering

**DBSCAN** groups together points that are closely packed, and marks points in low-density regions as noise/outliers (labeled `-1`). `eps` controls how close two points need to be to count as neighbors, and `min_samples` is the minimum number of neighbors needed for a point to count as a "core" point of a cluster.

In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5)  # Adjust eps and min_samples as needed
clusters = dbscan.fit_predict(df[numeric_columns])

# Add cluster labels to the DataFrame
df['DBSCAN_Cluster'] = clusters

# Analyze the results (e.g., count the number of points in each cluster)
print(df['DBSCAN_Cluster'].value_counts())

With these default settings, DBSCAN puts nearly every country in one giant cluster and flags a handful as noise — not very informative. **HDBSCAN** (Hierarchical DBSCAN) is a more flexible variant that doesn't require picking a single fixed `eps`; instead it examines the data's density structure at multiple scales, which often finds more meaningful, varied cluster sizes. It's not part of scikit-learn, so we install it separately.

In [ ]:
!pip install hdbscan

Run HDBSCAN, using `min_cluster_size` to control the smallest group size it's willing to call a genuine cluster (rather than noise).

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size=5) # Adjust min_cluster_size as needed
clusterer.fit(df[numeric_columns])
df['HDBSCAN_cluster'] = clusterer.labels_
df

Count how many countries fell into each HDBSCAN cluster (`-1` again means "noise", not assigned to any cluster).

In [ ]:
print(df['HDBSCAN_cluster'].value_counts())

Visualize the clusters along two economically meaningful axes: GDP per capita and literacy rate.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))  # Adjust figure size as needed
plt.scatter(df['GDP ($ per capita)'], df['Literacy (%)'], c=df['HDBSCAN_cluster'], cmap='viridis', s=50)
plt.xlabel('GDP ($ per capita)')
plt.ylabel('Literacy (%)')
plt.title('Cluster Plot (HDBSCAN)')
plt.colorbar(label='Cluster Label')
plt.show()

List the distinct cluster labels HDBSCAN found.

In [ ]:
unique_hdb_clusters = df['HDBSCAN_cluster'].unique()
unique_hdb_clusters

Finally, print out which countries ended up in each cluster, to see if the groupings make intuitive sense.

In [ ]:
cluster_groups = df.groupby('HDBSCAN_cluster')

# Iterate through each cluster and print the countries
for cluster_label, cluster_data in cluster_groups:
    print(f"Cluster {cluster_label}:")
    countries = cluster_data['Country'].tolist()
    print(countries)
    print()

## Task

Apply k-Means and k-Medoid Clustering to `df[numeric_columns]`, the same way DBSCAN and HDBSCAN were applied above. For k-Means, use `sklearn.cluster.KMeans` — you'll need to pick a number of clusters (`n_clusters`) up front, unlike DBSCAN/HDBSCAN which infer it automatically; try the "elbow method" (plotting inertia against several values of `n_clusters`) to help choose one. For k-Medoids, look into `sklearn_extra.cluster.KMedoids` (`pip install scikit-learn-extra`), which works similarly to k-Means but picks actual data points as cluster centers instead of computed means — this makes it more robust to outliers. Compare the resulting clusters (and the countries in each) to what DBSCAN and HDBSCAN found above.

In [ ]:
# Write your k-Means and k-Medoids clustering code here

## Practice Problems

1. Try a few different `eps` values for DBSCAN (e.g. 0.2, 0.3, 0.8) with `min_samples=5`. How sensitive is the number of clusters (and how many points get marked as noise, `-1`) to this choice?
2. Try a few different `min_cluster_size` values for HDBSCAN (e.g. 3, 10, 15). Do the resulting clusters and the printed country lists change substantially?
3. Instead of `MinMaxScaler`, try `StandardScaler` for the preprocessing step. Re-run HDBSCAN — does the choice of scaler change which countries end up grouped together?
4. The imputation step filled missing values with each column's *global* mean. Try instead filling missing values using the mean *within each `Region`* (hint: `df.groupby('Region')[numeric_columns].transform('mean')`). Does this more localized imputation change the clustering results?
5. Pick one HDBSCAN cluster from the printed country lists and research what those countries actually have in common economically or geographically. Does the clustering match your intuition?